# UC-GCP-1 — Resumable Upload via `init-upload` + Ingestion

**Persona:** Data Engineer

**Goal:** Demonstrate the full GCS-backed lifecycle using the production
resumable-upload flow — the same one a real browser/CLI client uses:

1. Create a catalog (`POST /stac/catalogs`) → the `gcp_provision` task creates a GCS bucket.
2. Create a collection.
3. `POST /gcp/buckets/catalogs/{cat}/collections/{col}/init-upload` returns a signed resumable `PUT` URL.
4. `PUT` the file bytes directly to the signed URL (no DynaStore hop).
5. Register the uploaded object as a collection **asset** so the virtual STAC collection-by-asset view can filter by `asset_id`.
6. Enable feature tracking.
7. Run the `ingestion` OGC process with `GcsDetailedReporter` writing reports back to the same bucket.
8. Verify features via `/features`, virtual STAC via `/stac/virtual/assets/{asset}/...`.
9. Hard-delete the catalog → bucket cleaned up.

**Contrast with `assets/01`:** that one uses the higher-level
`POST /assets/.../upload` (UploadTicket abstraction). This one uses the GCP-
specific `init-upload` endpoint directly, which is what the `gcp_bucket`
extension exposes for browser clients.

**Prerequisites:**
- GeoID platform reachable at `DYNASTORE_BASE_URL`
- `GCPModule` + `gcp_bucket` extension active
- A sysadmin JWT in `DYNASTORE_TOKEN` (creating catalogs + running processes requires it)

**Env vars:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`.  
**Optional:** `RUN_ID` (defaults to a random suffix).

In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN", "")
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:10])

CATALOG_ID = f"nb_{RUN_ID}"
COLLECTION_ID = f"col_{RUN_ID}"
ASSET_ID = f"asset_{RUN_ID}"
FILENAME = f"{ASSET_ID}.geojson"
CONTENT_TYPE = "application/geo+json"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)
print(f"Connected to {BASE_URL}")
print(f"Catalog={CATALOG_ID}  Collection={COLLECTION_ID}  Asset={ASSET_ID}")

## Step 1 — Create the catalog

Creating a catalog fires the `gcp_provision_catalog` lifecycle task, which
creates a deterministically-named GCS bucket. Managed eventing (Pub/Sub push
subscriptions) is disabled by default when `K_SERVICE` is absent — a localhost
target cannot receive push callbacks, so `OBJECT_FINALIZE` events will NOT
fire automatically. We register the asset manually in step 5 to compensate.

In [ ]:
catalog_payload = {
    "id": CATALOG_ID,
    "title": f"Notebook demo {RUN_ID}",
    "description": "Init-upload + ingestion demo",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
print(r.status_code, r.text[:200])
assert r.status_code in (200, 201), f"catalog create: {r.text}"

# Wait for the bucket provisioning task to finish.
for attempt in range(30):
    r = client.get(f"/features/catalogs/{CATALOG_ID}")
    cat = r.json() if r.status_code == 200 else {}
    status = cat.get("provisioning_status") or cat.get("status")
    if status == "ready":
        break
    time.sleep(2)
else:
    print("WARNING: catalog not reported 'ready' after 60s; continuing anyway")
print(f"Catalog provisioned: {CATALOG_ID}")

## Step 2 — Create the collection

In [ ]:
collection_payload = {
    "type": "Collection",
    "id": COLLECTION_ID,
    "title": "Init-upload demo collection",
    "description": "Features ingested from a GCS artifact uploaded via init-upload",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [[None, None]]},
    },
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
print(r.status_code, r.text[:200])
assert r.status_code in (200, 201), f"collection create: {r.text}"
print(f"Collection created: {COLLECTION_ID}")

## Step 3 — `init-upload` → signed resumable PUT URL

`POST /gcp/buckets/catalogs/{cat}/collections/{col}/init-upload` takes an
`InitiateUploadRequest` body (`filename`, `content_type`, `asset`, optional
`upload_options`) and returns an `InitiateUploadResponse`:

- `upload_id` — opaque handle for this session
- `upload_uri` — a signed GCS resumable-upload URL to `PUT` bytes to
- `status` — `"initiated"`

The signed URL bypasses the DynaStore API server entirely — the client
streams bytes straight to GCS. This is the flow the web uploader uses in
production.

In [ ]:
init_payload = {
    "filename": FILENAME,
    "content_type": CONTENT_TYPE,
    "asset": {
        "asset_id": ASSET_ID,
        "asset_type": "VECTORIAL",
        "metadata": {"source": "notebook-demo"},
    },
    "upload_options": {
        "predefined_acl": "publicRead",
        "if_generation_match": 0,  # refuse to overwrite an existing object
    },
}
r = client.post(
    f"/gcp/buckets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/init-upload",
    content=json.dumps(init_payload),
)
print(r.status_code, r.text[:300])
assert r.status_code == 200, f"init-upload: {r.text}"

init_resp = r.json()
upload_uri = init_resp["upload_uri"]
upload_id = init_resp["upload_id"]
print(f"\nupload_id   : {upload_id}")
print(f"upload_uri  : {upload_uri[:80]}…")

## Step 4 — PUT the file bytes to the signed URL

This is the actual upload. Real clients stream from a file handle; we build
a small GeoJSON `Feature` in-memory to keep the notebook self-contained.

In [ ]:
feature = {
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
    "properties": {
        "id": f"feat_{RUN_ID}",
        "datetime": "2024-01-01T00:00:00Z",
        "prop": "demo-value",
    },
}
body = json.dumps(feature).encode("utf-8")

# A separate client: the signed URL has its own auth (the signature) — no
# DynaStore bearer token is needed, and sending one causes GCS to reject it.
upload_client = httpx.Client(timeout=120.0)
resp = upload_client.put(
    upload_uri,
    content=body,
    headers={"Content-Type": CONTENT_TYPE},
)
print(resp.status_code, resp.text[:200])
assert resp.status_code in (200, 201), f"PUT to signed URL failed: {resp.text}"
print(f"Uploaded {len(body)} bytes via signed URL")
upload_client.close()

## Step 5 — Register the asset

In production an `OBJECT_FINALIZE` Pub/Sub event fires on the notification
channel and the platform auto-registers the asset. Locally (no push
subscription) we register it manually. The asset URI is derived from the
catalog's bucket name plus the object path the init-upload session targeted.

In [ ]:
# The init-upload response doesn't echo the final gs:// URI explicitly —
# we derive it from the catalog's bucket identifier.
r = client.get(f"/gcp/buckets/catalogs/{CATALOG_ID}")
print(r.status_code, r.text[:200])
assert r.status_code == 200, f"bucket lookup: {r.text}"
bucket_info = r.json()
bucket_name = bucket_info.get("bucket_id") or bucket_info.get("bucket_name")
assert bucket_name, f"no bucket name in response: {bucket_info}"

# init-upload puts objects under {collection_id}/{filename} by default
object_path = f"{COLLECTION_ID}/{FILENAME}"
asset_uri = f"gs://{bucket_name}/{object_path}"
print(f"Asset URI: {asset_uri}")

register_payload = {
    "asset_id": ASSET_ID,
    "asset_type": "ASSET",
    "uri": asset_uri,
    "metadata": {"asset_href": f"/proxy/r/{ASSET_ID}"},
}
r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    content=json.dumps(register_payload),
)
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"asset register: {r.text}"
print(f"Asset {ASSET_ID} registered → {asset_uri}")

## Step 6 — Enable feature tracking on the collection

In [ ]:
stac_config = {
    "asset_tracking": {
        "enabled": True,
        "access_mode": "proxy",
    }
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/StacPluginConfig",
    content=json.dumps(stac_config),
)
print(r.status_code, r.text[:200])
assert r.status_code in (200, 204), f"config put: {r.text}"

## Step 7 — Run the `ingestion` process

Synchronous execution via `Prefer: wait=true`. The `GcsDetailedReporter`
config points at a path on the **same bucket** — after completion the report
blob shows up under `gs://{bucket}/ingestion-reports/`.

In [ ]:
ingest_inputs = {
    "catalog_id": CATALOG_ID,
    "collection_id": COLLECTION_ID,
    "ingestion_request": {
        "asset": {
            "asset_id": ASSET_ID,
            "uri": asset_uri,
            "metadata": {},
        },
        "column_mapping": {
            "external_id": "id",
            "attributes_source_type": "explicit_list",
            "attribute_mapping": [
                {"source": "prop", "map_to": "prop"},
                {"source": "datetime", "map_to": "datetime"},
            ],
        },
        "time_validity_start_column": "datetime",
        "source_srid": 4326,
        "encoding": "utf-8",
        "read_batch_size": 1000,
        "database_batch_size": 1000,
        "reporting": {
            "GcsDetailedReporter": {
                "report_file_path": f"gs://{bucket_name}/ingestion-reports/{{task_id}}-{{timestamp_utc}}.json",
                "output_format": "JSON",
                "report_content": "ALL",
            }
        },
    },
}

r = client.post(
    f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
    content=json.dumps({"inputs": ingest_inputs, "outputs": {}}),
    headers={**headers, "Prefer": "wait=true"},
)
print(r.status_code, r.text[:400])
assert r.status_code in (200, 201), f"ingestion exec: {r.text}"
job = r.json()
assert job.get("status") == "successful", f"ingestion failed: {job}"
print(f"Ingestion job {job.get('jobID')} → successful")

## Step 8 — Verify features + virtual STAC collection-by-asset

In [ ]:
# 8a — features endpoint
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
print(r.status_code, r.text[:300])
assert r.status_code == 200, f"features list: {r.text}"
feats = r.json().get("features", [])
assert len(feats) >= 1, "features endpoint returned no items"
print(f"Features endpoint: {len(feats)} item(s)")

# 8b — virtual STAC collection-by-asset
r = client.get(
    f"/stac/virtual/assets/{ASSET_ID}/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
)
print(r.status_code, r.text[:200])
assert r.status_code == 200
assert r.json()["id"] == ASSET_ID

r = client.get(
    f"/stac/virtual/assets/{ASSET_ID}/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
)
assert r.status_code == 200
vfeats = r.json().get("features", [])
assert len(vfeats) >= 1, "virtual STAC returned no items"
print(f"Virtual STAC collection-by-asset: {len(vfeats)} item(s)")

## Teardown — hard-delete catalog

`force=true` bypasses soft-delete. The destruction hook fires
`gcp_catalog_cleanup_task` which deletes the bucket and all its objects,
including the uploaded artifact and the ingestion report.

In [ ]:
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
)
print(r.status_code, r.text[:200])
assert r.status_code in (200, 204, 404), f"hard-delete: {r.text}"
print(f"Catalog {CATALOG_ID} hard-deleted")
client.close()

## Why this matters

- `init-upload` returns a **signed URL** — the file bytes never touch
  DynaStore, reducing ingress cost and load on the API server.
- The upload_id is opaque, but the PUT-complete event on the bucket triggers
  the platform's asset auto-registration (when managed eventing is enabled).
- `if_generation_match=0` in `upload_options` prevents overwriting an
  existing object — GCS returns 412 Precondition Failed, which the API
  surfaces as HTTP 400 with a clear message.
- `predefined_acl="publicRead"` is applied to the uploaded object; swap to
  `"private"` for catalogs with access-controlled assets.

**See also:**
- `assets/01_register_asset_and_gcs_upload.ipynb` — the higher-level
  `UploadTicket` flow (backend-agnostic).
- `tests/dynastore/modules/gcp/integration/test_gcp_e2e_lifecycle.py` — the
  same flow exercised as a pytest integration test (uses the direct GCS
  client instead of `init-upload` to keep the test hermetic).